# glass-0-1-6_vs_5: current SVM/KNN/RF/LR comparison

Methods: no resampling, SMOTE, BSMOTE, ROS, ADASYN, CASB, CASB-relaxed, CASB-edited, CASB-ROS. RUS, SMOTEENN and MCSMOTE are excluded. Historical records remain in metadata.

CASB uses its original interpolation generator; CASB-ROS duplicates sources selected with CASB's original 50/30/20 quotas. This modular extension is inspired by the UGO paper, not a reproduction of UGO. Five supplied folds, seeds 42/137, same training-only ratio search (0.5/0.75/1) maximizing (G-mean+F1)/2. RBF SVM C=1 gamma=scale; distance-weighted KNN k=min(7,resampled count). RF: 100 trees, minimum leaf size 2; LR: C=1, max_iter=3000. These classifiers were retained after inspecting benchmark results, so the expanded comparison is exploratory. All preprocessing is fitted on training data. Abalone uses one-hot encoding.

Run all cells to inspect cached evidence. RF/LR evidence is imported from progress.md; GaussianNB remains in that report. Use the shared data_splits, choose_ratio and evaluate_fit functions in binary_oversampling.py to refit any saved setting. ADASYN failures are reported without fallback; incomplete method averages are withheld. All datasets have been inspected during development; this is exploratory evidence.


In [1]:
import sys, os
from pathlib import Path
sys.dont_write_bytecode=True
sys.path.insert(0,str(Path(os.environ.get('TEMP','/tmp'))/'ml_project_runtime'))
sys.path.insert(0,str(Path.cwd()))
import pandas as pd
from IPython.display import display, Markdown
from binary_oversampling import current_dataset_records, CURRENT_METHODS
import json
nb=json.loads(Path('16_glass-0-1-6_vs_5_CASB.ipynb').read_text(encoding='utf8'))
records=nb['metadata']['current_comparison']['records']
classifiers=['svm', 'knn', 'rf', 'lr']
frame=pd.DataFrame(records)
assert len(frame)==360
if 'error' not in frame:frame['error']=None
errors=frame[frame.error.notna()]
print('Attempted:',len(frame),'Failed:',len(errors))
if len(errors):display(errors[['classifier','method','seed','fold','error']])
frame['composite']=(frame.gmean+frame.f1)/2
metrics=['gmean','f1','precision','recall','specificity','ba','ap','auc','composite']


Attempted: 360 Failed: 0


In [2]:
ok=frame[frame.error.isna()]
counts=ok.groupby(['classifier','method']).size()
summary=ok.groupby(['classifier','method'])[metrics].mean()
for key in summary.index:
    if counts.loc[key]!=10:summary.loc[key,:]=float('nan')
summary=summary.reindex(pd.MultiIndex.from_product([classifiers,CURRENT_METHODS],names=['classifier','method']))
for clf in classifiers:
    display(Markdown('## '+clf.upper()+': averages (%)'))
    display((100*summary.loc[clf]).round(3))
combined=summary.groupby('method').mean().reindex(CURRENT_METHODS)
for method in CURRENT_METHODS:
    if summary.xs(method,level='method').isna().any().any():combined.loc[method,:]=float('nan')
display(Markdown('## Equal average of SVM, KNN, RF and LR (%)'))
display((100*combined).round(3))
print('NaN indicates incomplete coverage, not a zero score.')


## SVM: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,34.142,33.333,40.000,30.0,100.000,65.000,76.190,96.571,33.738
smote,73.651,66.000,63.333,70.0,98.857,84.429,79.500,98.000,69.825
bsmote,73.651,66.000,63.333,70.0,98.857,84.429,79.500,98.000,69.825
ros,73.359,63.333,60.000,70.0,98.286,84.143,80.167,97.857,68.346
adasyn,73.651,66.000,63.333,70.0,98.857,84.429,81.500,98.143,69.825
casb,73.651,66.000,63.333,70.0,98.857,84.429,75.667,97.714,69.825
casb_relaxed,73.651,66.000,63.333,70.0,98.857,84.429,77.333,98.000,69.825
casb_edited,73.359,63.333,60.000,70.0,98.286,84.143,82.333,98.286,68.346
casb_ros,73.359,63.333,60.000,70.0,98.286,84.143,77.167,97.571,68.346


## KNN: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,68.081,63.333,70.000,60.0,99.429,79.714,87.207,94.286,65.707
smote,98.845,85.333,76.667,100.0,97.714,98.857,80.833,98.929,92.089
bsmote,98.845,85.333,76.667,100.0,97.714,98.857,80.833,98.929,92.089
ros,92.927,76.000,68.333,90.0,97.429,93.714,78.041,93.929,84.464
adasyn,98.557,78.667,66.667,100.0,97.143,98.571,76.667,98.857,88.612
casb,93.071,79.333,73.333,90.0,97.714,93.857,75.541,93.786,86.202
casb_relaxed,93.071,79.333,73.333,90.0,97.714,93.857,74.500,98.286,86.202
casb_edited,92.927,76.000,68.333,90.0,97.429,93.714,74.500,98.286,84.464
casb_ros,93.071,79.333,73.333,90.0,97.714,93.857,75.541,93.857,86.202


## RF: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,46.927,44.667,46.667,45.0,99.714,72.357,92.500,99.286,45.797
smote,76.783,72.667,73.333,75.0,98.857,86.929,90.000,99.286,74.725
bsmote,76.783,72.667,73.333,75.0,98.857,86.929,90.000,99.286,74.725
ros,99.568,94.000,90.000,100.0,99.143,99.571,91.667,99.429,96.784
adasyn,80.866,74.667,76.667,75.0,98.857,86.929,85.833,99.000,77.766
casb,73.896,69.667,71.667,70.0,99.143,84.571,88.333,99.143,71.782
casb_relaxed,76.682,71.000,68.333,75.0,98.571,86.786,85.833,99.000,73.841
casb_edited,69.712,66.000,63.333,70.0,99.143,84.571,87.500,99.143,67.856
casb_ros,90.824,84.333,88.333,85.0,99.143,92.071,92.500,99.429,87.579


## LR: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,14.142,13.333,20.000,10.0,99.429,54.714,85.000,98.571,13.738
smote,97.977,72.000,56.667,100.0,96.000,98.000,88.333,99.143,84.988
bsmote,97.685,69.333,53.333,100.0,95.429,97.714,88.333,99.143,83.509
ros,97.977,72.000,56.667,100.0,96.000,98.000,88.333,99.143,84.988
adasyn,97.977,72.000,56.667,100.0,96.000,98.000,86.667,99.000,84.988
casb,97.977,72.000,56.667,100.0,96.000,98.000,85.000,98.857,84.988
casb_relaxed,97.831,70.667,55.000,100.0,95.714,97.857,86.667,99.000,84.249
casb_edited,97.829,71.048,55.667,100.0,95.714,97.857,86.667,99.000,84.438
casb_ros,97.977,72.000,56.667,100.0,96.000,98.000,85.000,98.857,84.988


## Equal average of SVM, KNN, RF and LR (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,40.823,38.667,44.167,36.25,99.643,67.946,85.224,97.179,39.745
smote,86.814,74.000,67.500,86.25,97.857,92.054,84.667,98.839,80.407
bsmote,86.741,73.333,66.667,86.25,97.714,91.982,84.667,98.839,80.037
ros,90.958,76.333,68.750,90.00,97.714,93.857,84.552,97.589,83.646
adasyn,87.763,72.833,65.833,86.25,97.714,91.982,82.667,98.750,80.298
casb,84.649,71.750,66.250,82.50,97.929,90.214,81.135,97.375,78.199
casb_relaxed,85.309,71.750,65.000,83.75,97.714,90.732,81.083,98.571,78.529
casb_edited,83.457,69.095,61.833,82.50,97.643,90.071,82.750,98.679,76.276
casb_ros,88.808,74.750,69.583,86.25,97.786,92.018,82.552,97.429,81.779


NaN indicates incomplete coverage, not a zero score.


In [3]:
core=['smote','bsmote','ros','adasyn']
family=['casb','casb_relaxed','casb_edited','casb_ros']
for view,table in [(clf,summary.loc[clf]) for clf in classifiers]+[('equal four-classifier average',combined)]:
    output=[]
    for metric in ['gmean','f1','precision','recall']:
        values=table.loc[core+family,metric]
        if values.isna().any():
            output.append(dict(metric=metric,result='Incomplete coverage'));continue
        ours=values.loc[family];base=values.loc[core];delta=ours.max()-base.max()
        output.append(dict(metric=metric,best_variant=ours.idxmax(),family_percent=100*ours.max(),baseline=base.idxmax(),baseline_percent=100*base.max(),difference_pp=100*delta,result='WIN' if delta>1e-12 else 'LOSS' if delta < -1e-12 else 'TIE'))
    display(Markdown('## '+view+': per-metric family comparison'))
    display(pd.DataFrame(output).round(3))
print('Retrospective per-metric family maxima; not one deployable winning method. No composite winner.')
display(Markdown('## Fold-level evidence'))
with pd.option_context('display.max_rows',400):
    display(frame[['seed','fold','classifier','method','ratio']+metrics+['error']])


## svm: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb,73.651,smote,73.651,0.0,TIE
1,f1,casb,66.000,smote,66.000,0.0,TIE
2,precision,casb,63.333,smote,63.333,0.0,TIE
3,recall,casb,70.000,smote,70.000,0.0,TIE


## knn: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb,93.071,smote,98.845,-5.774,LOSS
1,f1,casb,79.333,smote,85.333,-6.000,LOSS
2,precision,casb,73.333,smote,76.667,-3.333,LOSS
3,recall,casb,90.000,smote,100.000,-10.000,LOSS


## rf: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_ros,90.824,ros,99.568,-8.745,LOSS
1,f1,casb_ros,84.333,ros,94.000,-9.667,LOSS
2,precision,casb_ros,88.333,ros,90.000,-1.667,LOSS
3,recall,casb_ros,85.000,ros,100.000,-15.000,LOSS


... (3 outputs left out)

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_ros,88.808,ros,90.958,-2.150,LOSS
1,f1,casb_ros,74.750,ros,76.333,-1.583,LOSS
2,precision,casb_ros,69.583,ros,68.750,0.833,WIN
3,recall,casb_ros,86.250,ros,90.000,-3.750,LOSS


Retrospective per-metric family maxima; not one deployable winning method. No composite winner.


## Fold-level evidence

,seed,fold,classifier,method,ratio,gmean,f1,precision,recall,specificity,ba,ap,auc,composite,error
0,42,1,svm,original,0.50,0.000000,0.000000,0.000000,0.0,1.000000,0.500000,1.000000,1.000000,0.000000,None
1,42,1,svm,smote,0.50,1.000000,1.000000,1.000000,1.0,1.000000,1.000000,1.000000,1.000000,1.000000,None
2,42,1,svm,bsmote,0.50,1.000000,1.000000,1.000000,1.0,1.000000,1.000000,1.000000,1.000000,1.000000,None
3,42,1,svm,ros,0.50,1.000000,1.000000,1.000000,1.0,1.000000,1.000000,1.000000,1.000000,1.000000,None
4,42,1,svm,adasyn,0.50,1.000000,1.000000,1.000000,1.0,1.000000,1.000000,1.000000,1.000000,1.000000,None
5,42,1,svm,casb,0.50,1.000000,1.000000,1.000000,1.0,1.000000,1.000000,1.000000,1.000000,1.000000,None
6,42,1,svm,casb_relaxed,0.50,1.000000,1.000000,1.000000,1.0,1.000000,1.000000,1.000000,1.000000,1.000000,None
7,42,1,svm,casb_edited,0.50,1.000000,1.000000,1.000000,1.0,1.000000,1.000000,1.000000,1.000000,1.000000,None
8,42,1,svm,casb_ros,0.50,1.000000,1.000000,1.000000,1.0,1.000000,1.000000,1.000000,1.000000,1.000000,None
9,42,1,knn,original,0.50,1.000000,1.000000,1.000000,1.0,1.000000,1.000000,1.000000,1.000000,1.000000,None
